In [ ]:
from google.colab import drive

drive.mount("/content/drive")

print("Google Drive mounted successfully.")

In [ ]:
# ================================================================
# Vision Mamba (Vim) - corrected installation for Google Colab
# Tesla T4 / CUDA
#
# IMPORTANT:
#   1. selective_scan_cuda MUST be compiled.
#   2. Do NOT replace ext_modules with [].
#   3. setup.py is reset BEFORE the CUDA-architecture patch.
#   4. The broken selective_scan_interface.py indentation patch
#      from the previous version is NOT used.
# ================================================================

import os
import sys
import time
import inspect
import subprocess
import shutil
import importlib
import torch


assert torch.cuda.is_available(), (
    "GPU is not available. "
    "Go to Runtime > Change runtime type > T4 GPU."
)


def sh(cmd, env=None):
    result = subprocess.run(
        cmd,
        shell=True,
        executable="/bin/bash",
        env=env,
        capture_output=True,
        text=True,
    )

    output = result.stdout + "\n" + result.stderr

    if result.returncode != 0:
        print(output[-25000:])
        raise RuntimeError(f"Command failed:\n{cmd}")

    if result.stdout:
        print(result.stdout[-8000:])

    return result.stdout


# ------------------------------------------------
# GPU information
# ------------------------------------------------

GPU_NAME = torch.cuda.get_device_name(0)
CC = torch.cuda.get_device_capability(0)
CUDA_ARCH = f"{CC[0]}{CC[1]}"

print("GPU:", GPU_NAME)
print("CUDA compute capability:", CC)
print("Architecture:", f"sm_{CUDA_ARCH}")


# ------------------------------------------------
# Clean old/broken installation
# ------------------------------------------------

VIM = "/content/Vim"

print("\nRemoving previous broken Vim installation...")

if os.path.isdir(VIM):
    shutil.rmtree(VIM)

sh(
    "pip uninstall -y mamba-ssm causal-conv1d "
    "2>/dev/null || true"
)


# ------------------------------------------------
# Build dependencies
# ------------------------------------------------

print("\nInstalling build dependencies...")

sh(
    "pip install -q "
    "einops ninja packaging timm huggingface_hub "
    "transformers triton"
)


# ------------------------------------------------
# Fresh Vim repository
# ------------------------------------------------

print("\nCloning Vim repository...")

sh(
    f"git clone -q --depth 1 "
    f"https://github.com/hustvl/Vim.git {VIM}"
)


# ------------------------------------------------
# Reset setup.py FIRST.
# This is important because git checkout would otherwise
# erase the CUDA architecture patch.
# ------------------------------------------------

MAMBA_SETUP = f"{VIM}/mamba-1p1p1/setup.py"
CAUSAL_SETUP = f"{VIM}/causal-conv1d/setup.py"

print("\nResetting build files...")

sh(
    f"cd {VIM}/mamba-1p1p1 && "
    "git checkout -- setup.py"
)

sh(
    f"cd {VIM}/causal-conv1d && "
    "git checkout -- setup.py"
)


# ------------------------------------------------
# Patch CUDA architecture AFTER reset
# ------------------------------------------------

def patch_cuda_arch(setup_path):
    text = open(setup_path, "r", encoding="utf-8").read()

    if "VIM_CUDA_ARCH" in text:
        print("CUDA architecture patch already present:", setup_path)
        return

    anchor = (
        "    # HACK: The compiler flag "
        "-D_GLIBCXX_USE_CXX11_ABI"
    )

    if anchor not in text:
        raise RuntimeError(
            f"Could not find CUDA patch anchor in:\n{setup_path}"
        )

    patch = (
        '    _vim_arch = os.getenv("VIM_CUDA_ARCH")\n'
        '    if _vim_arch and not HIP_BUILD:\n'
        '        cc_flag = [\n'
        '            "-gencode",\n'
        '            f"arch=compute_{_vim_arch},'
        'code=sm_{_vim_arch}"\n'
        '        ]\n\n'
        + anchor
    )

    text = text.replace(anchor, patch, 1)

    with open(setup_path, "w", encoding="utf-8") as f:
        f.write(text)

    print("Patched:", setup_path)


patch_cuda_arch(MAMBA_SETUP)
patch_cuda_arch(CAUSAL_SETUP)


# ------------------------------------------------
# Transformers compatibility
# ------------------------------------------------

GENERATION = (
    f"{VIM}/mamba-1p1p1/"
    "mamba_ssm/utils/generation.py"
)

g = open(GENERATION, "r", encoding="utf-8").read()

old_import = (
    "from transformers.generation import "
    "GreedySearchDecoderOnlyOutput, "
    "SampleDecoderOnlyOutput, TextStreamer"
)

if old_import in g:
    new_import = """try:
    from transformers.generation import (
        GreedySearchDecoderOnlyOutput,
        SampleDecoderOnlyOutput,
        TextStreamer,
    )
except ImportError:
    from transformers.generation import (
        GenerateDecoderOnlyOutput as GreedySearchDecoderOnlyOutput,
        GenerateDecoderOnlyOutput as SampleDecoderOnlyOutput,
        TextStreamer,
    )
"""
    g = g.replace(old_import, new_import, 1)

    with open(GENERATION, "w", encoding="utf-8") as f:
        f.write(g)

    print("Transformers compatibility patch applied.")


# ------------------------------------------------
# IMPORTANT:
# Do NOT modify selective_scan_interface.py.
#
# Vim imports:
#     import selective_scan_cuda
#
# Therefore the CUDA extension must actually be built.
# ------------------------------------------------

print(
    "\nselective_scan_interface.py is left unchanged."
)
print(
    "selective_scan_cuda will be compiled by mamba-1p1p1."
)


# ------------------------------------------------
# Build environment
# ------------------------------------------------

env = dict(
    os.environ,
    VIM_CUDA_ARCH=CUDA_ARCH,
    MAMBA_FORCE_BUILD="TRUE",
    CAUSAL_CONV1D_FORCE_BUILD="TRUE",
    MAX_JOBS=str(max(1, min(4, os.cpu_count() or 2))),
)

print("\nBuild configuration:")
print("GPU:", GPU_NAME)
print("CUDA architecture:", f"sm_{CUDA_ARCH}")
print("MAX_JOBS:", env["MAX_JOBS"])


# ------------------------------------------------
# Build causal-conv1d
# ------------------------------------------------

t0 = time.time()

print("\n" + "=" * 70)
print("BUILDING causal-conv1d")
print("=" * 70)

sh(
    f"cd {VIM}/causal-conv1d && "
    "pip install --verbose "
    "--no-build-isolation "
    "--no-deps "
    ".",
    env,
)

print(
    f"\ncausal-conv1d completed in "
    f"{(time.time() - t0) / 60:.2f} minutes"
)


# ------------------------------------------------
# Build Mamba.
#
# DO NOT disable ext_modules.
# This build must produce selective_scan_cuda.
# ------------------------------------------------

print("\n" + "=" * 70)
print("BUILDING mamba_ssm + selective_scan_cuda")
print("=" * 70)

sh(
    f"cd {VIM}/mamba-1p1p1 && "
    "pip install --verbose "
    "--no-build-isolation "
    "--no-deps "
    ".",
    env,
)

print(
    f"\nMamba completed in "
    f"{(time.time() - t0) / 60:.2f} minutes"
)


# ------------------------------------------------
# Clear stale imports
# ------------------------------------------------

for key in list(sys.modules.keys()):
    if (
        "mamba_ssm" in key
        or "causal_conv1d" in key
        or "selective_scan_cuda" in key
    ):
        sys.modules.pop(key, None)

importlib.invalidate_caches()


# ------------------------------------------------
# Verify selective_scan_cuda FIRST
# ------------------------------------------------

print("\n" + "=" * 70)
print("VERIFYING selective_scan_cuda")
print("=" * 70)

try:
    import selective_scan_cuda

    print(
        "SUCCESS: selective_scan_cuda imported."
    )
    print(
        "Location:",
        selective_scan_cuda.__file__
    )

except Exception:
    import traceback
    traceback.print_exc()
    raise RuntimeError(
        "selective_scan_cuda was not built correctly."
    )


# ------------------------------------------------
# Verify causal_conv1d
# ------------------------------------------------

print("\n" + "=" * 70)
print("VERIFYING causal_conv1d")
print("=" * 70)

try:
    import causal_conv1d
    import causal_conv1d_cuda

    print(
        "SUCCESS: causal_conv1d CUDA extension imported."
    )
    print(
        "Location:",
        causal_conv1d_cuda.__file__
    )

except Exception:
    import traceback
    traceback.print_exc()
    raise RuntimeError(
        "causal_conv1d CUDA extension was not built correctly."
    )


# ------------------------------------------------
# Verify mamba_ssm
# ------------------------------------------------

print("\n" + "=" * 70)
print("VERIFYING mamba_ssm")
print("=" * 70)

try:
    import mamba_ssm
    from mamba_ssm.modules.mamba_simple import Mamba

    print(
        "SUCCESS: mamba_ssm imported."
    )
    print(
        "Location:",
        mamba_ssm.__file__
    )

except Exception:
    import traceback
    traceback.print_exc()
    raise


# ------------------------------------------------
# Verify Vim-specific Mamba API
# ------------------------------------------------

signature = inspect.signature(Mamba.__init__)
parameters = list(signature.parameters.keys())

print("\nMamba parameters:")
print(parameters)

if "bimamba_type" not in parameters:
    raise RuntimeError(
        "The installed Mamba does not contain "
        "'bimamba_type'. The Vim-compatible Mamba "
        "implementation was not installed."
    )

print("\n" + "=" * 70)
print("VIM KERNELS READY")
print("=" * 70)
print("GPU                  :", GPU_NAME)
print("CUDA architecture    :", f"sm_{CUDA_ARCH}")
print("causal_conv1d        : OK")
print("causal_conv1d_cuda   : OK")
print("selective_scan_cuda  : OK")
print("mamba_ssm            : OK")
print("bimamba_type         : OK")
print("\nVision Mamba dependencies are ready.")


In [ ]:
# ===== Vision Mamba (Vim) architecture: copied UNCHANGED from vim/models_mamba.py =====
import sys
sys.path.insert(0, "/content/Vim/vim")   # only addition: so that `from rope import *` finds rope.py

# Copyright (c) 2015-present, Facebook, Inc.
# All rights reserved.
import torch
import torch.nn as nn
from functools import partial
from torch import Tensor
from typing import Optional

from timm.models.vision_transformer import VisionTransformer, _cfg
from timm.models.registry import register_model
from timm.models.layers import trunc_normal_, lecun_normal_

from timm.models.layers import DropPath, to_2tuple
from timm.models.vision_transformer import _load_weights

import math

from collections import namedtuple

from mamba_ssm.modules.mamba_simple import Mamba
from mamba_ssm.utils.generation import GenerationMixin
from mamba_ssm.utils.hf import load_config_hf, load_state_dict_hf

from rope import *
import random

try:
    from mamba_ssm.ops.triton.layer_norm import RMSNorm, layer_norm_fn, rms_norm_fn
except ImportError:
    RMSNorm, layer_norm_fn, rms_norm_fn = None, None, None


__all__ = [
    'vim_tiny_patch16_224', 'vim_small_patch16_224', 'vim_base_patch16_224',
    'vim_tiny_patch16_384', 'vim_small_patch16_384', 'vim_base_patch16_384',
]


class PatchEmbed(nn.Module):
    """ 2D Image to Patch Embedding
    """
    def __init__(self, img_size=224, patch_size=16, stride=16, in_chans=3, embed_dim=768, norm_layer=None, flatten=True):
        super().__init__()
        img_size = to_2tuple(img_size)
        patch_size = to_2tuple(patch_size)
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = ((img_size[0] - patch_size[0]) // stride + 1, (img_size[1] - patch_size[1]) // stride + 1)
        self.num_patches = self.grid_size[0] * self.grid_size[1]
        self.flatten = flatten

        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=stride)
        self.norm = norm_layer(embed_dim) if norm_layer else nn.Identity()

    def forward(self, x):
        B, C, H, W = x.shape
        assert H == self.img_size[0] and W == self.img_size[1], \
            f"Input image size ({H}*{W}) doesn't match model ({self.img_size[0]}*{self.img_size[1]})."
        x = self.proj(x)
        if self.flatten:
            x = x.flatten(2).transpose(1, 2)  # BCHW -> BNC
        x = self.norm(x)
        return x
    

class Block(nn.Module):
    def __init__(
        self, dim, mixer_cls, norm_cls=nn.LayerNorm, fused_add_norm=False, residual_in_fp32=False,drop_path=0.,
    ):
        """
        Simple block wrapping a mixer class with LayerNorm/RMSNorm and residual connection"

        This Block has a slightly different structure compared to a regular
        prenorm Transformer block.
        The standard block is: LN -> MHA/MLP -> Add.
        [Ref: https://arxiv.org/abs/2002.04745]
        Here we have: Add -> LN -> Mixer, returning both
        the hidden_states (output of the mixer) and the residual.
        This is purely for performance reasons, as we can fuse add and LayerNorm.
        The residual needs to be provided (except for the very first block).
        """
        super().__init__()
        self.residual_in_fp32 = residual_in_fp32
        self.fused_add_norm = fused_add_norm
        # import ipdb; ipdb.set_trace()
        self.mixer = mixer_cls(dim)
        self.norm = norm_cls(dim)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        if self.fused_add_norm:
            assert RMSNorm is not None, "RMSNorm import fails"
            assert isinstance(
                self.norm, (nn.LayerNorm, RMSNorm)
            ), "Only LayerNorm and RMSNorm are supported for fused_add_norm"

    def forward(
        self, hidden_states: Tensor, residual: Optional[Tensor] = None, inference_params=None
    ):
        r"""Pass the input through the encoder layer.

        Args:
            hidden_states: the sequence to the encoder layer (required).
            residual: hidden_states = Mixer(LN(residual))
        """
        if not self.fused_add_norm:
            if residual is None:
                residual = hidden_states
            else:
                residual = residual + self.drop_path(hidden_states)
            
            hidden_states = self.norm(residual.to(dtype=self.norm.weight.dtype))
            if self.residual_in_fp32:
                residual = residual.to(torch.float32)
        else:
            fused_add_norm_fn = rms_norm_fn if isinstance(self.norm, RMSNorm) else layer_norm_fn
            if residual is None:
                hidden_states, residual = fused_add_norm_fn(
                    hidden_states,
                    self.norm.weight,
                    self.norm.bias,
                    residual=residual,
                    prenorm=True,
                    residual_in_fp32=self.residual_in_fp32,
                    eps=self.norm.eps,
                )
            else:
                hidden_states, residual = fused_add_norm_fn(
                    self.drop_path(hidden_states),
                    self.norm.weight,
                    self.norm.bias,
                    residual=residual,
                    prenorm=True,
                    residual_in_fp32=self.residual_in_fp32,
                    eps=self.norm.eps,
                )    
        hidden_states = self.mixer(hidden_states, inference_params=inference_params)
        return hidden_states, residual

    def allocate_inference_cache(self, batch_size, max_seqlen, dtype=None, **kwargs):
        return self.mixer.allocate_inference_cache(batch_size, max_seqlen, dtype=dtype, **kwargs)


def create_block(
    d_model,
    d_state=16,
    ssm_cfg=None,
    norm_epsilon=1e-5,
    drop_path=0.,
    rms_norm=False,
    residual_in_fp32=False,
    fused_add_norm=False,
    layer_idx=None,
    device=None,
    dtype=None,
    if_bimamba=False,
    bimamba_type="none",
    if_divide_out=False,
    init_layer_scale=None,
):
    if if_bimamba:
        bimamba_type = "v1"
    if ssm_cfg is None:
        ssm_cfg = {}
    factory_kwargs = {"device": device, "dtype": dtype}
    # import ipdb; ipdb.set_trace()
    mixer_cls = partial(Mamba, d_state=d_state, layer_idx=layer_idx, bimamba_type=bimamba_type, if_divide_out=if_divide_out, init_layer_scale=init_layer_scale, **ssm_cfg, **factory_kwargs)
    norm_cls = partial(
        nn.LayerNorm if not rms_norm else RMSNorm, eps=norm_epsilon, **factory_kwargs
    )
    block = Block(
        d_model,
        mixer_cls,
        norm_cls=norm_cls,
        drop_path=drop_path,
        fused_add_norm=fused_add_norm,
        residual_in_fp32=residual_in_fp32,
    )
    block.layer_idx = layer_idx
    return block


# https://github.com/huggingface/transformers/blob/c28d04e9e252a1a099944e325685f14d242ecdcd/src/transformers/models/gpt2/modeling_gpt2.py#L454
def _init_weights(
    module,
    n_layer,
    initializer_range=0.02,  # Now only used for embedding layer.
    rescale_prenorm_residual=True,
    n_residuals_per_layer=1,  # Change to 2 if we have MLP
):
    if isinstance(module, nn.Linear):
        if module.bias is not None:
            if not getattr(module.bias, "_no_reinit", False):
                nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        nn.init.normal_(module.weight, std=initializer_range)

    if rescale_prenorm_residual:
        # Reinitialize selected weights subject to the OpenAI GPT-2 Paper Scheme:
        #   > A modified initialization which accounts for the accumulation on the residual path with model depth. Scale
        #   > the weights of residual layers at initialization by a factor of 1/√N where N is the # of residual layers.
        #   >   -- GPT-2 :: https://openai.com/blog/better-language-models/
        #
        # Reference (Megatron-LM): https://github.com/NVIDIA/Megatron-LM/blob/main/megatron/model/gpt_model.py
        for name, p in module.named_parameters():
            if name in ["out_proj.weight", "fc2.weight"]:
                # Special Scaled Initialization --> There are 2 Layer Norms per Transformer Block
                # Following Pytorch init, except scale by 1/sqrt(2 * n_layer)
                # We need to reinit p since this code could be called multiple times
                # Having just p *= scale would repeatedly scale it down
                nn.init.kaiming_uniform_(p, a=math.sqrt(5))
                with torch.no_grad():
                    p /= math.sqrt(n_residuals_per_layer * n_layer)


def segm_init_weights(m):
    if isinstance(m, nn.Linear):
        trunc_normal_(m.weight, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.Conv2d):
        # NOTE conv was left to pytorch default in my original init
        lecun_normal_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, (nn.LayerNorm, nn.GroupNorm, nn.BatchNorm2d)):
        nn.init.zeros_(m.bias)
        nn.init.ones_(m.weight)


class VisionMamba(nn.Module):
    def __init__(self, 
                 img_size=224, 
                 patch_size=16, 
                 stride=16,
                 depth=24, 
                 embed_dim=192, 
                 d_state=16,
                 channels=3, 
                 num_classes=1000,
                 ssm_cfg=None, 
                 drop_rate=0.,
                 drop_path_rate=0.1,
                 norm_epsilon: float = 1e-5, 
                 rms_norm: bool = True, 
                 initializer_cfg=None,
                 fused_add_norm=True,
                 residual_in_fp32=True,
                 device=None,
                 dtype=None,
                 ft_seq_len=None,
                 pt_hw_seq_len=14,
                 if_bidirectional=False,
                 final_pool_type='none',
                 if_abs_pos_embed=True,
                 if_rope=False,
                 if_rope_residual=False,
                 flip_img_sequences_ratio=-1.,
                 if_bimamba=False,
                 bimamba_type="v2",
                 if_cls_token=True,
                 if_divide_out=True,
                 init_layer_scale=None,
                 use_double_cls_token=False,
                 use_middle_cls_token=True,
                 **kwargs):
        factory_kwargs = {"device": device, "dtype": dtype}
        # add factory_kwargs into kwargs
        kwargs.update(factory_kwargs) 
        super().__init__()
        self.residual_in_fp32 = residual_in_fp32
        self.fused_add_norm = fused_add_norm
        self.if_bidirectional = if_bidirectional
        self.final_pool_type = final_pool_type
        self.if_abs_pos_embed = if_abs_pos_embed
        self.if_rope = if_rope
        self.if_rope_residual = if_rope_residual
        self.flip_img_sequences_ratio = flip_img_sequences_ratio
        self.if_cls_token = if_cls_token
        self.use_double_cls_token = use_double_cls_token
        self.use_middle_cls_token = use_middle_cls_token
        self.num_tokens = 1 if if_cls_token else 0

        # pretrain parameters
        self.num_classes = num_classes
        self.d_model = self.num_features = self.embed_dim = embed_dim  # num_features for consistency with other models

        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, stride=stride, in_chans=channels, embed_dim=embed_dim)
        num_patches = self.patch_embed.num_patches

        if if_cls_token:
            if use_double_cls_token:
                self.cls_token_head = nn.Parameter(torch.zeros(1, 1, self.embed_dim))
                self.cls_token_tail = nn.Parameter(torch.zeros(1, 1, self.embed_dim))
                self.num_tokens = 2
            else:
                self.cls_token = nn.Parameter(torch.zeros(1, 1, self.embed_dim))
                # self.num_tokens = 1
            
        if if_abs_pos_embed:
            self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + self.num_tokens, self.embed_dim))
            self.pos_drop = nn.Dropout(p=drop_rate)

        if if_rope:
            half_head_dim = embed_dim // 2
            hw_seq_len = img_size // patch_size
            self.rope = VisionRotaryEmbeddingFast(
                dim=half_head_dim,
                pt_seq_len=pt_hw_seq_len,
                ft_seq_len=hw_seq_len
            )
        self.head = nn.Linear(self.num_features, num_classes) if num_classes > 0 else nn.Identity()


        # TODO: release this comment
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]  # stochastic depth decay rule
        # import ipdb;ipdb.set_trace()
        inter_dpr = [0.0] + dpr
        self.drop_path = DropPath(drop_path_rate) if drop_path_rate > 0. else nn.Identity()
                # transformer blocks
        self.layers = nn.ModuleList(
            [
                create_block(
                    embed_dim,
                    d_state=d_state,
                    ssm_cfg=ssm_cfg,
                    norm_epsilon=norm_epsilon,
                    rms_norm=rms_norm,
                    residual_in_fp32=residual_in_fp32,
                    fused_add_norm=fused_add_norm,
                    layer_idx=i,
                    if_bimamba=if_bimamba,
                    bimamba_type=bimamba_type,
                    drop_path=inter_dpr[i],
                    if_divide_out=if_divide_out,
                    init_layer_scale=init_layer_scale,
                    **factory_kwargs,
                )
                for i in range(depth)
            ]
        )
        
        # output head
        self.norm_f = (nn.LayerNorm if not rms_norm else RMSNorm)(
            embed_dim, eps=norm_epsilon, **factory_kwargs
        )

        # self.pre_logits = nn.Identity()

        # original init
        self.patch_embed.apply(segm_init_weights)
        self.head.apply(segm_init_weights)
        if if_abs_pos_embed:
            trunc_normal_(self.pos_embed, std=.02)
        if if_cls_token:
            if use_double_cls_token:
                trunc_normal_(self.cls_token_head, std=.02)
                trunc_normal_(self.cls_token_tail, std=.02)
            else:
                trunc_normal_(self.cls_token, std=.02)

        # mamba init
        self.apply(
            partial(
                _init_weights,
                n_layer=depth,
                **(initializer_cfg if initializer_cfg is not None else {}),
            )
        )


    def allocate_inference_cache(self, batch_size, max_seqlen, dtype=None, **kwargs):
        return {
            i: layer.allocate_inference_cache(batch_size, max_seqlen, dtype=dtype, **kwargs)
            for i, layer in enumerate(self.layers)
        }

    @torch.jit.ignore
    def no_weight_decay(self):
        return {"pos_embed", "cls_token", "dist_token", "cls_token_head", "cls_token_tail"}

    @torch.jit.ignore()
    def load_pretrained(self, checkpoint_path, prefix=""):
        _load_weights(self, checkpoint_path, prefix)

    def forward_features(self, x, inference_params=None, if_random_cls_token_position=False, if_random_token_rank=False):
        # taken from https://github.com/rwightman/pytorch-image-models/blob/master/timm/models/vision_transformer.py
        # with slight modifications to add the dist_token
        x = self.patch_embed(x)
        B, M, _ = x.shape

        if self.if_cls_token:
            if self.use_double_cls_token:
                cls_token_head = self.cls_token_head.expand(B, -1, -1)
                cls_token_tail = self.cls_token_tail.expand(B, -1, -1)
                token_position = [0, M + 1]
                x = torch.cat((cls_token_head, x, cls_token_tail), dim=1)
                M = x.shape[1]
            else:
                if self.use_middle_cls_token:
                    cls_token = self.cls_token.expand(B, -1, -1)
                    token_position = M // 2
                    # add cls token in the middle
                    x = torch.cat((x[:, :token_position, :], cls_token, x[:, token_position:, :]), dim=1)
                elif if_random_cls_token_position:
                    cls_token = self.cls_token.expand(B, -1, -1)
                    token_position = random.randint(0, M)
                    x = torch.cat((x[:, :token_position, :], cls_token, x[:, token_position:, :]), dim=1)
                    print("token_position: ", token_position)
                else:
                    cls_token = self.cls_token.expand(B, -1, -1)  # stole cls_tokens impl from Phil Wang, thanks
                    token_position = 0
                    x = torch.cat((cls_token, x), dim=1)
                M = x.shape[1]

        if self.if_abs_pos_embed:
            # if new_grid_size[0] == self.patch_embed.grid_size[0] and new_grid_size[1] == self.patch_embed.grid_size[1]:
            #     x = x + self.pos_embed
            # else:
            #     pos_embed = interpolate_pos_embed_online(
            #                 self.pos_embed, self.patch_embed.grid_size, new_grid_size,0
            #             )
            x = x + self.pos_embed
            x = self.pos_drop(x)

        if if_random_token_rank:

            # 生成随机 shuffle 索引
            shuffle_indices = torch.randperm(M)

            if isinstance(token_position, list):
                print("original value: ", x[0, token_position[0], 0], x[0, token_position[1], 0])
            else:
                print("original value: ", x[0, token_position, 0])
            print("original token_position: ", token_position)

            # 执行 shuffle
            x = x[:, shuffle_indices, :]

            if isinstance(token_position, list):
                # 找到 cls token 在 shuffle 之后的新位置
                new_token_position = [torch.where(shuffle_indices == token_position[i])[0].item() for i in range(len(token_position))]
                token_position = new_token_position
            else:
                # 找到 cls token 在 shuffle 之后的新位置
                token_position = torch.where(shuffle_indices == token_position)[0].item()

            if isinstance(token_position, list):
                print("new value: ", x[0, token_position[0], 0], x[0, token_position[1], 0])
            else:
                print("new value: ", x[0, token_position, 0])
            print("new token_position: ", token_position)




        if_flip_img_sequences = False
        if self.flip_img_sequences_ratio > 0 and (self.flip_img_sequences_ratio - random.random()) > 1e-5:
            x = x.flip([1])
            if_flip_img_sequences = True

        # mamba impl
        residual = None
        hidden_states = x
        if not self.if_bidirectional:
            for layer in self.layers:

                if if_flip_img_sequences and self.if_rope:
                    hidden_states = hidden_states.flip([1])
                    if residual is not None:
                        residual = residual.flip([1])

                # rope about
                if self.if_rope:
                    hidden_states = self.rope(hidden_states)
                    if residual is not None and self.if_rope_residual:
                        residual = self.rope(residual)

                if if_flip_img_sequences and self.if_rope:
                    hidden_states = hidden_states.flip([1])
                    if residual is not None:
                        residual = residual.flip([1])

                hidden_states, residual = layer(
                    hidden_states, residual, inference_params=inference_params
                )
        else:
            # get two layers in a single for-loop
            for i in range(len(self.layers) // 2):
                if self.if_rope:
                    hidden_states = self.rope(hidden_states)
                    if residual is not None and self.if_rope_residual:
                        residual = self.rope(residual)

                hidden_states_f, residual_f = self.layers[i * 2](
                    hidden_states, residual, inference_params=inference_params
                )
                hidden_states_b, residual_b = self.layers[i * 2 + 1](
                    hidden_states.flip([1]), None if residual == None else residual.flip([1]), inference_params=inference_params
                )
                hidden_states = hidden_states_f + hidden_states_b.flip([1])
                residual = residual_f + residual_b.flip([1])

        if not self.fused_add_norm:
            if residual is None:
                residual = hidden_states
            else:
                residual = residual + self.drop_path(hidden_states)
            hidden_states = self.norm_f(residual.to(dtype=self.norm_f.weight.dtype))
        else:
            # Set prenorm=False here since we don't need the residual
            fused_add_norm_fn = rms_norm_fn if isinstance(self.norm_f, RMSNorm) else layer_norm_fn
            hidden_states = fused_add_norm_fn(
                self.drop_path(hidden_states),
                self.norm_f.weight,
                self.norm_f.bias,
                eps=self.norm_f.eps,
                residual=residual,
                prenorm=False,
                residual_in_fp32=self.residual_in_fp32,
            )

        # return only cls token if it exists
        if self.if_cls_token:
            if self.use_double_cls_token:
                return (hidden_states[:, token_position[0], :] + hidden_states[:, token_position[1], :]) / 2
            else:
                if self.use_middle_cls_token:
                    return hidden_states[:, token_position, :]
                elif if_random_cls_token_position:
                    return hidden_states[:, token_position, :]
                else:
                    return hidden_states[:, token_position, :]

        if self.final_pool_type == 'none':
            return hidden_states[:, -1, :]
        elif self.final_pool_type == 'mean':
            return hidden_states.mean(dim=1)
        elif self.final_pool_type == 'max':
            return hidden_states
        elif self.final_pool_type == 'all':
            return hidden_states
        else:
            raise NotImplementedError

    def forward(self, x, return_features=False, inference_params=None, if_random_cls_token_position=False, if_random_token_rank=False):
        x = self.forward_features(x, inference_params, if_random_cls_token_position=if_random_cls_token_position, if_random_token_rank=if_random_token_rank)
        if return_features:
            return x
        x = self.head(x)
        if self.final_pool_type == 'max':
            x = x.max(dim=1)[0]
        return x


@register_model
def vim_tiny_patch16_224_bimambav2_final_pool_mean_abs_pos_embed_with_midclstok_div2(pretrained=False, **kwargs):
    model = VisionMamba(
        patch_size=16, embed_dim=192, depth=24, rms_norm=True, residual_in_fp32=True, fused_add_norm=True, final_pool_type='mean', if_abs_pos_embed=True, if_rope=False, if_rope_residual=False, bimamba_type="v2", if_cls_token=True, if_divide_out=True, use_middle_cls_token=True, **kwargs)
    model.default_cfg = _cfg()
    if pretrained:
        checkpoint = torch.hub.load_state_dict_from_url(
            url="to.do",
            map_location="cpu", check_hash=True
        )
        model.load_state_dict(checkpoint["model"])
    return model

@register_model
def vim_tiny_patch16_stride8_224_bimambav2_final_pool_mean_abs_pos_embed_with_midclstok_div2(pretrained=False, **kwargs):
    model = VisionMamba(
        patch_size=16, stride=8, embed_dim=192, depth=24, rms_norm=True, residual_in_fp32=True, fused_add_norm=True, final_pool_type='mean', if_abs_pos_embed=True, if_rope=False, if_rope_residual=False, bimamba_type="v2", if_cls_token=True, if_divide_out=True, use_middle_cls_token=True, **kwargs)
    model.default_cfg = _cfg()
    if pretrained:
        checkpoint = torch.hub.load_state_dict_from_url(
            url="to.do",
            map_location="cpu", check_hash=True
        )
        model.load_state_dict(checkpoint["model"])
    return model

@register_model
def vim_small_patch16_224_bimambav2_final_pool_mean_abs_pos_embed_with_midclstok_div2(pretrained=False, **kwargs):
    model = VisionMamba(
        patch_size=16, embed_dim=384, depth=24, rms_norm=True, residual_in_fp32=True, fused_add_norm=True, final_pool_type='mean', if_abs_pos_embed=True, if_rope=False, if_rope_residual=False, bimamba_type="v2", if_cls_token=True, if_divide_out=True, use_middle_cls_token=True, **kwargs)
    model.default_cfg = _cfg()
    if pretrained:
        checkpoint = torch.hub.load_state_dict_from_url(
            url="to.do",
            map_location="cpu", check_hash=True
        )
        model.load_state_dict(checkpoint["model"])
    return model

@register_model
def vim_small_patch16_stride8_224_bimambav2_final_pool_mean_abs_pos_embed_with_midclstok_div2(pretrained=False, **kwargs):
    model = VisionMamba(
        patch_size=16, stride=8, embed_dim=384, depth=24, rms_norm=True, residual_in_fp32=True, fused_add_norm=True, final_pool_type='mean', if_abs_pos_embed=True, if_rope=False, if_rope_residual=False, bimamba_type="v2", if_cls_token=True, if_divide_out=True, use_middle_cls_token=True, **kwargs)
    model.default_cfg = _cfg()
    if pretrained:
        checkpoint = torch.hub.load_state_dict_from_url(
            url="to.do",
            map_location="cpu", check_hash=True
        )
        model.load_state_dict(checkpoint["model"])
    return model
    
@register_model
def vim_base_patch16_224_bimambav2_final_pool_mean_abs_pos_embed_with_middle_cls_token_div2(pretrained=False, **kwargs):
    model = VisionMamba(
        patch_size=16, embed_dim=768, d_state=16, depth=24, rms_norm=True, residual_in_fp32=True, fused_add_norm=True, final_pool_type='mean', if_abs_pos_embed=True, if_rope=False, if_rope_residual=False, bimamba_type="v2", if_cls_token=True, if_devide_out=True, use_middle_cls_token=True, **kwargs)
    model.default_cfg = _cfg()
    if pretrained:
        checkpoint = torch.hub.load_state_dict_from_url(
            url="to.do",
            map_location="cpu", check_hash=True
        )
        model.load_state_dict(checkpoint["model"])
    return model

In [ ]:
import os
import time
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_fscore_support
)

# Device select pannrom
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

In [ ]:
# Dataset paths set pannrom
BASE_DIR = "/content/drive/MyDrive/dataset/MILK10k_Split"

# BASE_DIR not found aanaa Drive la train/val/test folder iruka nu search pannrom
if not os.path.isdir(os.path.join(BASE_DIR, "train")):
    print("Not found at:", BASE_DIR, "- searching Drive ...")
    for dp, dn, _ in os.walk("/content/drive/MyDrive"):
        if dp[len("/content/drive/MyDrive"):].count(os.sep) >= 5:
            dn[:] = []; continue
        if {"train", "val", "test"}.issubset(set(dn)):
            BASE_DIR = dp; break
    print("Using BASE_DIR:", BASE_DIR)

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR = os.path.join(BASE_DIR, "val")
TEST_DIR = os.path.join(BASE_DIR, "test")

TRAIN_GT = os.path.join(TRAIN_DIR, "GroundTruth.csv")
VAL_GT = os.path.join(VAL_DIR, "GroundTruth.csv")
TEST_GT = os.path.join(TEST_DIR, "GroundTruth.csv")

TRAIN_IMG_DIR = os.path.join(TRAIN_DIR, "images")
VAL_IMG_DIR = os.path.join(VAL_DIR, "images")
TEST_IMG_DIR = os.path.join(TEST_DIR, "images")

print("Train directory exists :", os.path.exists(TRAIN_DIR))
print("Val directory exists   :", os.path.exists(VAL_DIR))
print("Test directory exists  :", os.path.exists(TEST_DIR))

print()
print("Train CSV exists :", os.path.exists(TRAIN_GT))
print("Val CSV exists   :", os.path.exists(VAL_GT))
print("Test CSV exists  :", os.path.exists(TEST_GT))

In [ ]:
# Classes configure pannrom
CLASS_NAMES = [
    "melanoma",
    "nevus",
    "bcc",
    "scc",
    "keratoses"
]

cls2idx = {
    "melanoma": 0,
    "nevus": 1,
    "bcc": 2,
    "scc": 3,
    "keratoses": 4
}

NUM_CLASSES = 5

print("Classes:")
for i, name in enumerate(CLASS_NAMES):
    print(i, "->", name)

In [ ]:
# Image transform set pannrom
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Image transform created.")
print("Input size: 224 x 224")

In [ ]:
# Skin dataset class create pannrom
class SkinDataset(Dataset):
    def __init__(self, split_dir, transform=None):
        self.split_dir = split_dir
        self.transform = transform

        self.csv_path = os.path.join(split_dir, "GroundTruth.csv")
        self.metadata_path = os.path.join(split_dir, "Metadata.csv")
        self.image_dir = os.path.join(split_dir, "images")

        self.df = pd.read_csv(self.csv_path)
        self.metadata = pd.read_csv(self.metadata_path)

        self.samples = []

        print()
        print("CSV path      :", self.csv_path)
        print("Metadata path :", self.metadata_path)
        print("Image path    :", self.image_dir)
        print("Image folder exists :", os.path.exists(self.image_dir))

        print()
        print("GroundTruth columns:")
        print(self.df.columns.tolist())

        print()
        print("Metadata columns:")
        print(self.metadata.columns.tolist())

        image_files = []

        for root, dirs, files in os.walk(self.image_dir):
            for file in files:
                if file.lower().endswith((".jpg", ".jpeg", ".png")):
                    image_files.append(os.path.join(root, file))

        print()
        print("Image files found :", len(image_files))

        image_lookup = {}

        for image_path in image_files:
            filename = os.path.basename(image_path)
            isic_id = os.path.splitext(filename)[0]
            image_lookup[isic_id] = image_path

        metadata_lookup = {}

        for _, row in self.metadata.iterrows():
            lesion_id = str(row["lesion_id"])
            isic_id = str(row["isic_id"])

            if lesion_id not in metadata_lookup:
                metadata_lookup[lesion_id] = []

            metadata_lookup[lesion_id].append(isic_id)

        matched_count = 0

        for _, row in self.df.iterrows():
            lesion_id = str(row["lesion_id"])

            isic_ids = metadata_lookup.get(lesion_id, [])

            if row["MEL"] == 1:
                label = cls2idx["melanoma"]
            elif row["NV"] == 1:
                label = cls2idx["nevus"]
            elif row["BCC"] == 1:
                label = cls2idx["bcc"]
            elif row["SCCKA"] == 1:
                label = cls2idx["scc"]
            elif row["BKL"] == 1 or row["AKIEC"] == 1:
                label = cls2idx["keratoses"]
            else:
                continue

            for isic_id in isic_ids:
                image_path = image_lookup.get(isic_id)

                if image_path is not None:
                    self.samples.append((image_path, label))
                    matched_count += 1

        print()
        print(f"Matched images : {matched_count}")
        print(f"Loaded {len(self.samples)} images from {self.csv_path}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, label = self.samples[index]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
# VisionMamba ku image transforms set pannrom
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Transforms created successfully.")

In [ ]:
# Datasets create pannrom
train_dataset = SkinDataset(
    split_dir=TRAIN_DIR,
    transform=train_transform
)

val_dataset = SkinDataset(
    split_dir=VAL_DIR,
    transform=val_transform
)

test_dataset = SkinDataset(
    split_dir=TEST_DIR,
    transform=val_transform
)

print()
print("Datasets created")
print("Train images :", len(train_dataset))
print("Val images   :", len(val_dataset))
print("Test images  :", len(test_dataset))

In [ ]:
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=False
)


In [ ]:
# Dataloader check pannrom
images, labels = next(iter(train_loader))

print("Image shape :", images.shape)
print("Label shape :", labels.shape)
print("Labels      :", labels[:10])

In [ ]:
# VisionMamba model create pannrom  (model functions come from models_mamba.py above - unchanged)
VIM_MODEL      = "small"      # "tiny" (115 MB weights)  or  "small" (413 MB weights)
USE_PRETRAINED = True         # ImageNet-1K pretrained Vim weights from HuggingFace (hustvl)
IMG_SIZE       = 224          # Vim PatchEmbed asserts 224 x 224 input

VIM_REGISTRY = {
    "tiny":  ("vim_tiny_patch16_224_bimambav2_final_pool_mean_abs_pos_embed_with_midclstok_div2",
              "hustvl/Vim-tiny-midclstok",  "vim_t_midclstok_76p1acc.pth"),
    "small": ("vim_small_patch16_224_bimambav2_final_pool_mean_abs_pos_embed_with_midclstok_div2",
              "hustvl/Vim-small-midclstok", "vim_s_midclstok_80p5acc.pth"),
}
fn_name, hf_repo, hf_file = VIM_REGISTRY[VIM_MODEL]

def build_vim():
    return globals()[fn_name](pretrained=False, num_classes=NUM_CLASSES, img_size=IMG_SIZE)

model = build_vim()

INIT = "scratch"
if USE_PRETRAINED:
    try:
        from huggingface_hub import hf_hub_download
        ckpt_file = hf_hub_download(repo_id=hf_repo, filename=hf_file)
        ckpt = torch.load(ckpt_file, map_location="cpu", weights_only=False)
        state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
        state = {k: v for k, v in state.items() if not k.startswith("head.")}      # new 5-class head
        own = model.state_dict()
        matched = sum(1 for k, v in state.items() if k in own and own[k].shape == v.shape)
        model.load_state_dict(state, strict=False)
        print(f"Loaded {matched}/{len(own)} tensors from {hf_repo}/{hf_file} (head is new)")
        INIT = "imagenet-1k"
    except Exception as e:
        print("!! Could not load pretrained weights:", repr(e))

model = model.to(DEVICE)

print("VisionMamba model created:", VIM_MODEL)
print(">>> INITIALISATION USED:", INIT.upper(), "<<<")
if INIT == "scratch":
    print("!! Training from random weights will give lower accuracy.")
print("Device      :", DEVICE)
print("Classes     :", NUM_CLASSES)
print("Input size  : 224 x 224")
print(f"Params      : {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

In [ ]:
# Checkpoint paths set pannrom
CHECKPOINT_DIR = (
    "/content/drive/MyDrive/"
    "VisionMamba_MILK10k_Checkpoints"
)

LAST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "last_checkpoint.pth"
)

BEST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "best_checkpoint.pth"
)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("Checkpoint status")
print("Last checkpoint :", LAST_CHECKPOINT)
print("Last exists     :", os.path.exists(LAST_CHECKPOINT))
print("Best checkpoint :", BEST_CHECKPOINT)
print("Best exists     :", os.path.exists(BEST_CHECKPOINT))

In [ ]:
# Last checkpoint load pannrom (irundha mattum resume, illana fresh training)
checkpoint = None

if os.path.exists(LAST_CHECKPOINT):
    checkpoint = torch.load(
        LAST_CHECKPOINT,
        map_location=DEVICE,
        weights_only=False
    )

    if checkpoint.get("vim_model", VIM_MODEL) != VIM_MODEL:
        print("!! Checkpoint is for Vim-" + str(checkpoint.get("vim_model")) +
              " but VIM_MODEL is '" + VIM_MODEL + "' - ignoring it (starting fresh)")
        checkpoint = None
    else:
        model.load_state_dict(
            checkpoint["model_state_dict"]
        )

        model = model.to(DEVICE)

        print("Checkpoint restored")
        print("Last completed epoch :", checkpoint["epoch"])
        print("Best Validation Macro-F1 :", checkpoint["best_f1"])
else:
    print("No checkpoint found - starting fresh training from epoch 1")

In [ ]:
# Loss function set pannrom
criterion = nn.CrossEntropyLoss()

print("Criterion:", criterion)

In [ ]:
# Optimizer and AMP scaler set pannrom
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0001
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=torch.cuda.is_available()
)

print("Optimizer:", optimizer)
print("AMP scaler created.")

In [ ]:
# Resume settings set pannrom
EPOCHS = 100

if checkpoint is not None:
    start_epoch = checkpoint["epoch"]
    best_f1 = checkpoint["best_f1"]
    best_epoch = checkpoint.get("best_epoch", 0)
    history = checkpoint.get("history", [])

    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])
else:
    start_epoch = 0
    best_f1 = -1.0
    best_epoch = 0
    history = []

print("Resume information")
print("Last completed epoch :", start_epoch)
print("Next epoch           :", start_epoch + 1)
print("Total epochs         :", EPOCHS)
print("Remaining epochs     :", EPOCHS - start_epoch)
print("Best Val Macro-F1    :", best_f1)
print("Best epoch so far    :", best_epoch)

In [ ]:
# Training function create pannrom
def train_one_epoch(epoch):
    model.train()

    running_loss = 0.0

    all_preds = []
    all_labels = []

    total_images = len(train_loader.dataset)

    progress_bar = tqdm(
        train_loader,
        desc=f"Training Epoch {epoch + 1}/{EPOCHS}",
        unit="batch"
    )

    for images, labels in progress_bar:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            enabled=torch.cuda.is_available()
        ):
            logits = model(images)

            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = images.size(0)

        running_loss += loss.item() * batch_size

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(
            preds.detach().cpu().numpy()
        )

        all_labels.extend(
            labels.detach().cpu().numpy()
        )

    epoch_loss = running_loss / total_images

    accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    balanced_accuracy = balanced_accuracy_score(
        all_labels,
        all_preds
    )

    macro_precision = precision_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    macro_recall = recall_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    weighted_precision = precision_score(
        all_labels,
        all_preds,
        average="weighted",
        zero_division=0
    )

    weighted_recall = recall_score(
        all_labels,
        all_preds,
        average="weighted",
        zero_division=0
    )

    weighted_f1 = f1_score(
        all_labels,
        all_preds,
        average="weighted",
        zero_division=0
    )

    precision_values, recall_values, f1_values, support_values = (
        precision_recall_fscore_support(
            all_labels,
            all_preds,
            labels=list(range(NUM_CLASSES)),
            zero_division=0
        )
    )

    per_class = {}

    for i, class_name in enumerate(CLASS_NAMES):
        per_class[class_name] = {
            "precision": precision_values[i],
            "recall": recall_values[i],
            "f1": f1_values[i],
            "support": int(support_values[i])
        }

    return {
        "loss": epoch_loss,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "weighted_precision": weighted_precision,
        "weighted_recall": weighted_recall,
        "weighted_f1": weighted_f1,
        "per_class": per_class
    }

print("train_one_epoch() created successfully.")

In [ ]:
# Validation and evaluation function create pannrom
def evaluate(loader):
    model.eval()

    running_loss = 0.0

    all_preds = []
    all_labels = []

    total_images = len(loader.dataset)

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            with torch.autocast(
                device_type="cuda",
                enabled=torch.cuda.is_available()
            ):
                outputs = model(images)

                if hasattr(outputs, "logits"):
                    logits = outputs.logits
                else:
                    logits = outputs

                loss = criterion(logits, labels)

            batch_size = images.size(0)

            running_loss += loss.item() * batch_size

            preds = torch.argmax(logits, dim=1)

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    epoch_loss = running_loss / total_images

    accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    balanced_accuracy = balanced_accuracy_score(
        all_labels,
        all_preds
    )

    macro_precision = precision_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    macro_recall = recall_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    weighted_precision = precision_score(
        all_labels,
        all_preds,
        average="weighted",
        zero_division=0
    )

    weighted_recall = recall_score(
        all_labels,
        all_preds,
        average="weighted",
        zero_division=0
    )

    weighted_f1 = f1_score(
        all_labels,
        all_preds,
        average="weighted",
        zero_division=0
    )

    precision_values, recall_values, f1_values, support_values = (
        precision_recall_fscore_support(
            all_labels,
            all_preds,
            labels=list(range(NUM_CLASSES)),
            zero_division=0
        )
    )

    per_class = {}

    for i, class_name in enumerate(CLASS_NAMES):
        per_class[class_name] = {
            "precision": precision_values[i],
            "recall": recall_values[i],
            "f1": f1_values[i],
            "support": int(support_values[i])
        }

    return {
        "loss": epoch_loss,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "weighted_precision": weighted_precision,
        "weighted_recall": weighted_recall,
        "weighted_f1": weighted_f1,
        "per_class": per_class
    }

print("evaluate() created successfully.")

In [ ]:
# Learning rate FIXED = 0.0001 (no scheduler, no decay)
LEARNING_RATE = 0.0001

for param_group in optimizer.param_groups:
    param_group["lr"] = LEARNING_RATE

print("Learning rate fixed at :", optimizer.param_groups[0]["lr"])
print("LR scheduler           : not used")

In [ ]:
# Final training setup check pannrom
print("VisionMamba training setup")
print("Device              :", DEVICE)
print("Train batches       :", len(train_loader))
print("Validation batches  :", len(val_loader))
print("Test batches        :", len(test_loader))
print("Start epoch         :", start_epoch)
print("Next epoch          :", start_epoch + 1)
print("Total epochs        :", EPOCHS)
print("Best Val Macro-F1   :", best_f1)

print()

print("Model               : OK")
print("Criterion           : OK")
print("Optimizer           : OK")
print("Scaler              : OK")
print("Learning rate       : fixed 0.0001")
print("train_one_epoch()   : OK")
print("evaluate()          : OK")

print()

print(
    "Last checkpoint exists :",
    os.path.exists(LAST_CHECKPOINT)
)

print(
    "Best checkpoint exists :",
    os.path.exists(BEST_CHECKPOINT)
)

In [ ]:
# Resume training pannrom
for epoch in range(
    start_epoch,
    EPOCHS
):
    print()
    print(f"Epoch {epoch + 1}/{EPOCHS}")

    start_time = time.time()

    # Training metrics calculate pannrom
    train_metrics = train_one_epoch(epoch)

    # Validation metrics calculate pannrom
    val_metrics = evaluate(val_loader)

    val_f1 = val_metrics["macro_f1"]

    epoch_time = time.time() - start_time

    # Training history update pannrom
    history.append({
        "epoch": epoch + 1,
        "train": train_metrics,
        "val": val_metrics,
        "time": epoch_time
    })

    print()
    print("TRAIN METRICS")

    print(
        f"Train Loss              : "
        f"{train_metrics['loss']:.4f}"
    )

    print(
        f"Train Accuracy          : "
        f"{train_metrics['accuracy']:.4f}"
    )

    print(
        f"Train Balanced Accuracy : "
        f"{train_metrics['balanced_accuracy']:.4f}"
    )

    print(
        f"Train Macro Precision   : "
        f"{train_metrics['macro_precision']:.4f}"
    )

    print(
        f"Train Macro Recall      : "
        f"{train_metrics['macro_recall']:.4f}"
    )

    print(
        f"Train Macro F1          : "
        f"{train_metrics['macro_f1']:.4f}"
    )

    print(
        f"Train Weighted Precision: "
        f"{train_metrics['weighted_precision']:.4f}"
    )

    print(
        f"Train Weighted Recall   : "
        f"{train_metrics['weighted_recall']:.4f}"
    )

    print(
        f"Train Weighted F1       : "
        f"{train_metrics['weighted_f1']:.4f}"
    )

    print()

    for class_name in CLASS_NAMES:
        m = train_metrics["per_class"][class_name]

        print(
            f"{class_name:<15} | "
            f"Precision: {m['precision']:.4f} | "
            f"Recall: {m['recall']:.4f} | "
            f"F1: {m['f1']:.4f} | "
            f"Support: {m['support']}"
        )

    print()
    print("VALIDATION METRICS")

    print(
        f"Validation Loss              : "
        f"{val_metrics['loss']:.4f}"
    )

    print(
        f"Validation Accuracy          : "
        f"{val_metrics['accuracy']:.4f}"
    )

    print(
        f"Validation Balanced Accuracy : "
        f"{val_metrics['balanced_accuracy']:.4f}"
    )

    print(
        f"Validation Macro Precision   : "
        f"{val_metrics['macro_precision']:.4f}"
    )

    print(
        f"Validation Macro Recall      : "
        f"{val_metrics['macro_recall']:.4f}"
    )

    print(
        f"Validation Macro F1          : "
        f"{val_metrics['macro_f1']:.4f}"
    )

    print(
        f"Validation Weighted Precision: "
        f"{val_metrics['weighted_precision']:.4f}"
    )

    print(
        f"Validation Weighted Recall   : "
        f"{val_metrics['weighted_recall']:.4f}"
    )

    print(
        f"Validation Weighted F1       : "
        f"{val_metrics['weighted_f1']:.4f}"
    )

    print()

    for class_name in CLASS_NAMES:
        m = val_metrics["per_class"][class_name]

        print(
            f"{class_name:<15} | "
            f"Precision: {m['precision']:.4f} | "
            f"Recall: {m['recall']:.4f} | "
            f"F1: {m['f1']:.4f} | "
            f"Support: {m['support']}"
        )

    # Best model check pannrom
    is_best = val_f1 > best_f1

    if is_best:
        best_f1 = val_f1
        best_epoch = epoch + 1

    # Last checkpoint save pannrom
    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_f1": best_f1,
        "best_epoch": best_epoch,
        "history": history,
        "class_names": CLASS_NAMES,
        "vim_model": VIM_MODEL
    }

    torch.save(
        checkpoint,
        LAST_CHECKPOINT
    )

    # Best checkpoint save pannrom
    if is_best:
        torch.save(
            checkpoint,
            BEST_CHECKPOINT
        )

        print()
        print("New best model saved")
        print(
            f"Best Validation Macro-F1 : "
            f"{best_f1:.4f}"
        )
        print(f"Best Epoch               : {epoch + 1}")

    print()
    print(
        f"Epoch time: "
        f"{epoch_time / 60:.2f} minutes"
    )
    print(f"Learning rate: {optimizer.param_groups[0]['lr']}")

# ===================== TRAINING FINISHED : best epoch summary =====================
if len(history) > 0:
    best_entry = max(
        history,
        key=lambda h: h["val"]["macro_f1"]
    )

    bv = best_entry["val"]
    bt = best_entry["train"]

    print()
    print("=" * 80)
    print(
        f"Epoch {best_entry['epoch']} has the best Macro-F1 value "
        f"on the validation set : {bv['macro_f1']:.4f}"
    )
    print("=" * 80)
    print(f"Best epoch                   : {best_entry['epoch']} / {EPOCHS}")
    print(f"Validation Loss              : {bv['loss']:.4f}")
    print(f"Validation Accuracy          : {bv['accuracy']:.4f}")
    print(f"Validation Balanced Accuracy : {bv['balanced_accuracy']:.4f}")
    print(f"Validation Macro Precision   : {bv['macro_precision']:.4f}")
    print(f"Validation Macro Recall      : {bv['macro_recall']:.4f}")
    print(f"Validation Macro F1          : {bv['macro_f1']:.4f}")
    print(f"Train Macro F1 (same epoch)  : {bt['macro_f1']:.4f}")
    print()

    for class_name in CLASS_NAMES:
        m = bv["per_class"][class_name]
        print(
            f"{class_name:<15} | "
            f"Precision: {m['precision']:.4f} | "
            f"Recall: {m['recall']:.4f} | "
            f"F1: {m['f1']:.4f} | "
            f"Support: {m['support']}"
        )

    print()
    print(f"The best checkpoint (epoch {best_entry['epoch']}) is saved at: {BEST_CHECKPOINT}")

In [ ]:
# Best VisionMamba model load pannrom
BEST_CHECKPOINT = (
    "/content/drive/MyDrive/"
    "VisionMamba_MILK10k_Checkpoints/"
    "best_checkpoint.pth"
)

print("Loading best VisionMamba model")
print("Checkpoint :", BEST_CHECKPOINT)
print(
    "Exists     :",
    os.path.exists(BEST_CHECKPOINT)
)

# VisionMamba model create pannrom (same architecture function as training)
model = build_vim().to(DEVICE)

# Best checkpoint load pannrom
checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print()
print("Best VisionMamba model loaded")
print("Checkpoint :", BEST_CHECKPOINT)
print("Epoch      :", checkpoint.get("epoch"), "<- this epoch has the best validation Macro-F1")
print("Best F1    :", checkpoint.get("best_f1"))
print("Model      :", "VisionMamba (Vim-" + VIM_MODEL + ")")

In [ ]:
# VisionMamba test evaluation pannrom
import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

model.eval()

all_predictions = []
all_labels = []

test_loss = 0.0
total_samples = 0

criterion = torch.nn.CrossEntropyLoss()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        outputs = model(images)

        if isinstance(outputs, tuple):
            outputs = outputs[0]

        loss = criterion(
            outputs,
            labels
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        batch_size = labels.size(0)

        test_loss += loss.item() * batch_size
        total_samples += batch_size

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)

test_loss = test_loss / total_samples

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

balanced_accuracy = balanced_accuracy_score(
    all_labels,
    all_predictions
)

macro_precision = precision_score(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
)

weighted_precision = precision_score(
    all_labels,
    all_predictions,
    average="weighted",
    zero_division=0
)

weighted_recall = recall_score(
    all_labels,
    all_predictions,
    average="weighted",
    zero_division=0
)

weighted_f1 = f1_score(
    all_labels,
    all_predictions,
    average="weighted",
    zero_division=0
)

print()
print("VisionMamba test results")
print(f"Test Loss               : {test_loss:.4f}")
print(f"Accuracy                : {accuracy:.4f}")
print(f"Balanced Accuracy       : {balanced_accuracy:.4f}")
print(f"Macro Precision         : {macro_precision:.4f}")
print(f"Macro Recall            : {macro_recall:.4f}")
print(f"Macro F1                : {macro_f1:.4f}")
print(f"Weighted Precision      : {weighted_precision:.4f}")
print(f"Weighted Recall         : {weighted_recall:.4f}")
print(f"Weighted F1             : {weighted_f1:.4f}")
print(f"Checkpoint epoch (best val Macro-F1): {checkpoint.get('epoch')}")

In [ ]:
# Per-class test metrics calculate pannrom
report = classification_report(
    all_labels,
    all_predictions,
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
)

print("Classification report")
print(report)

In [ ]:
# Confusion matrix create pannrom
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("Confusion Matrix:")
print(cm)

plt.figure(figsize=(8, 8))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=CLASS_NAMES
)

disp.plot(
    cmap="Blues",
    values_format="d"
)

plt.title(
    "VisionMamba - Test Confusion Matrix"
)

plt.xlabel("Predicted Class")
plt.ylabel("True Class")

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
# Normalized confusion matrix create pannrom
cm_normalized = confusion_matrix(
    all_labels,
    all_predictions,
    normalize="true"
)

plt.figure(figsize=(8, 8))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_normalized,
    display_labels=CLASS_NAMES
)

disp.plot(
    cmap="Blues",
    values_format=".2f"
)

plt.title(
    "VisionMamba - Normalized Test Confusion Matrix"
)

plt.xlabel("Predicted Class")
plt.ylabel("True Class")

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
# Ovvoru class-kum oru test image select pannrom
class_images = {}

for index in range(len(test_dataset)):
    image_path, label = test_dataset.samples[index]
    class_name = CLASS_NAMES[label]

    if class_name not in class_images:
        class_images[class_name] = {
            "index": index,
            "path": image_path,
            "label": label
        }

    if len(class_images) == len(CLASS_NAMES):
        break

print("Test image selected for each class")

for class_name in CLASS_NAMES:
    item = class_images[class_name]

    print()
    print("Class :", class_name)
    print("Index :", item["index"])
    print("Path  :", item["path"])

In [ ]:
# VisionMamba Grad-CAM for all 5 classes

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

# Test image select pannrom
IMAGE_INDEX = 100

IMAGE_PATH = test_dataset.samples[IMAGE_INDEX][0]
TRUE_LABEL = test_dataset.samples[IMAGE_INDEX][1]

print("Image:", IMAGE_PATH)
print("True Class:", CLASS_NAMES[TRUE_LABEL])

# Model eval mode la set pannrom
model.eval()

# VisionMamba target layer select pannrom
# The last block's mixer INPUT tokens are used. (The last block's output is not usable: the head reads only the
# class token, so patch-token gradients at the output would be exactly zero.)
target_layer = model.layers[-1].mixer

# Class token position (middle-cls-token design) and patch grid
grid_h, grid_w = model.patch_embed.grid_size
cls_position = model.patch_embed.num_patches // 2

# Image transform pannrom
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Original image load pannrom
original_image = Image.open(
    IMAGE_PATH
).convert("RGB")

image_tensor = transform(
    original_image
).unsqueeze(0).to(DEVICE)

# Activations and gradients store pannrom
activations = None
gradients = None

def save_gradient(grad):
    global gradients
    gradients = grad

def forward_hook(module, input, output):
    global activations
    activations = input[0]                 # (1, 197, C) tokens entering the last Mamba mixer
    input[0].register_hook(save_gradient)

forward_handle = target_layer.register_forward_hook(
    forward_hook
)

# Forward pass pannrom
model.zero_grad()

output = model(image_tensor)

if hasattr(output, "logits"):
    logits = output.logits
elif isinstance(output, tuple):
    logits = output[0]
else:
    logits = output

# Prediction calculate pannrom
probabilities = torch.softmax(
    logits,
    dim=1
)

predicted_class = torch.argmax(
    probabilities,
    dim=1
).item()

predicted_confidence = probabilities[
    0,
    predicted_class
].item()

print("\nPrediction:", CLASS_NAMES[predicted_class])
print("Confidence:", f"{predicted_confidence:.2%}")

# All 5 classes-kum Grad-CAM create pannrom
cams = []

for class_index in range(NUM_CLASSES):
    model.zero_grad()
    gradients = None

    class_score = logits[
        0,
        class_index
    ]

    class_score.backward(
        retain_graph=True
    )

    activation = activations.detach()
    gradient = gradients.detach()

    # class token remove pannitu patch tokens mattum edukkrom
    activation = torch.cat([activation[:, :cls_position], activation[:, cls_position + 1:]], dim=1)
    gradient = torch.cat([gradient[:, :cls_position], gradient[:, cls_position + 1:]], dim=1)

    weights = gradient.mean(
        dim=1,
        keepdim=True
    )

    cam = (
        weights * activation
    ).sum(dim=-1)

    cam = torch.relu(cam)
    cam = cam.reshape(grid_h, grid_w).float().cpu().numpy()

    if cam.max() > cam.min():
        cam = (
            cam - cam.min()
        ) / (
            cam.max() - cam.min()
        )
    else:
        cam = np.zeros_like(cam)

    cam_image = Image.fromarray(
        np.uint8(cam * 255)
    )

    cam_image = cam_image.resize(
        original_image.size
    )

    cam = np.array(
        cam_image
    ) / 255.0

    cams.append(cam)

# All 5 Grad-CAMs display pannrom
plt.figure(figsize=(20, 10))

for i in range(NUM_CLASSES):
    plt.subplot(2, 3, i + 1)

    plt.imshow(original_image)

    plt.imshow(
        cams[i],
        cmap="jet",
        alpha=0.45
    )

    class_name = CLASS_NAMES[i]

    class_probability = probabilities[
        0,
        i
    ].item()

    plt.title(
        f"{class_name}\n"
        f"Probability: {class_probability:.2%}"
    )

    plt.axis("off")

# Input image display pannrom
plt.subplot(2, 3, 6)

plt.imshow(original_image)

plt.title(
    f"Input Image\n"
    f"True: {CLASS_NAMES[TRUE_LABEL]}\n"
    f"Predicted: {CLASS_NAMES[predicted_class]}"
)

plt.axis("off")

plt.suptitle(
    "VisionMamba - Grad-CAM for All 5 Classes",
    fontsize=18
)

plt.tight_layout()
plt.show()

# Grad-CAM results print pannrom
print("Grad-CAM Results - All 5 Classes")

print(
    "True Class     :",
    CLASS_NAMES[TRUE_LABEL]
)

print(
    "Predicted Class:",
    CLASS_NAMES[predicted_class]
)

print(
    "Confidence     :",
    f"{predicted_confidence:.4f}"
)

print("-" * 70)

for i in range(NUM_CLASSES):
    print(
        f"{CLASS_NAMES[i]:12s} : "
        f"{probabilities[0, i].item():.4f}"
    )

# Hooks remove pannrom
forward_handle.remove()

In [ ]:
# VisionMamba Integrated Gradients (explainable AI) for all 5 classes

import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter
from torchvision import transforms
from scipy.ndimage import gaussian_filter

# ---------- settings ----------
IMAGE_INDEX  = 100       # test image index (same default as the Grad-CAM cell)
N_STEPS      = 50        # more steps = more accurate integral (try 100 if the completeness error is high)
CHUNK        = 10        # path points per forward/backward pass (lower it if CUDA runs out of memory)
BASELINE     = "black"   # "black" | "mean" | "blur"
SMOOTH_SIGMA = 3         # heatmap smoothing for display only (0 = none)

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# Test image select pannrom
IMAGE_PATH = test_dataset.samples[IMAGE_INDEX][0]
TRUE_LABEL = test_dataset.samples[IMAGE_INDEX][1]

print("Image:", IMAGE_PATH)
print("True Class:", CLASS_NAMES[TRUE_LABEL])

# Image transform pannrom
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

original_image = Image.open(IMAGE_PATH).convert("RGB")
x = transform(original_image).unsqueeze(0).to(DEVICE)

# Baseline image select pannrom ("no information" reference image)
mean_t = torch.tensor(MEAN, device=DEVICE).view(1, 3, 1, 1)
std_t = torch.tensor(STD, device=DEVICE).view(1, 3, 1, 1)

if BASELINE == "black":
    baseline = (torch.zeros_like(x) - mean_t) / std_t
elif BASELINE == "mean":
    baseline = torch.zeros_like(x)
elif BASELINE == "blur":
    blurred = original_image.filter(ImageFilter.GaussianBlur(radius=25))
    baseline = transform(blurred).unsqueeze(0).to(DEVICE)
else:
    raise ValueError("BASELINE must be 'black', 'mean' or 'blur'")

model.eval()


def get_logits(inputs):
    out = model(inputs)
    if hasattr(out, "logits"):
        out = out.logits
    elif isinstance(out, tuple):
        out = out[0]
    return out


# Integrated Gradients function create pannrom
def integrated_gradients(x, baseline, target, n_steps=50, chunk=10):
    # midpoint Riemann sum of the gradients along the straight path baseline -> image
    alphas = (torch.arange(n_steps, device=x.device, dtype=torch.float32) + 0.5) / n_steps
    grad_sum = torch.zeros_like(x)

    for start in range(0, n_steps, chunk):
        a = alphas[start:start + chunk].view(-1, 1, 1, 1)
        path = (baseline + a * (x - baseline)).detach().requires_grad_(True)

        with torch.enable_grad():
            score = get_logits(path)[:, target].sum()
            grads = torch.autograd.grad(score, path)[0]

        grad_sum += grads.sum(dim=0, keepdim=True)

    return (x - baseline) * grad_sum / n_steps


# Prediction calculate pannrom
with torch.no_grad():
    logits_x = get_logits(x)[0]
    logits_base = get_logits(baseline)[0]

probabilities = torch.softmax(logits_x, dim=0)
predicted_class = int(torch.argmax(probabilities).item())
predicted_confidence = probabilities[predicted_class].item()

print("\nPrediction:", CLASS_NAMES[predicted_class])
print("Confidence:", f"{predicted_confidence:.2%}")

# All 5 classes-kum Integrated Gradients create pannrom
attributions = []

print()
print("Completeness check (sum of attributions should be close to F(image) - F(baseline))")
print(f"{'class':12s} | {'F(x)-F(base)':>13s} | {'sum(attr)':>10s} | {'abs. error':>10s} | {'rel. error':>10s}")

for c in range(NUM_CLASSES):
    attr = integrated_gradients(x, baseline, c, N_STEPS, CHUNK)
    attributions.append(attr.detach())

    expected = (logits_x[c] - logits_base[c]).item()
    got = attr.sum().item()
    abs_error = abs(got - expected)
    rel_error = abs_error / max(abs(expected), 0.1)      # floor 0.1 so tiny logit differences do not blow it up

    print(f"{CLASS_NAMES[c]:12s} | {expected:13.4f} | {got:10.4f} | {abs_error:10.4f} | {rel_error:10.2%}")

print("If the relative error is above ~10% for the predicted class, increase N_STEPS.")


# Attribution -> heatmap function
def to_heatmap(attr, positive_only=True):
    m = attr.sum(dim=1)[0].float().cpu().numpy()        # sum over R, G, B

    if positive_only:
        m = np.clip(m, 0, None)

    if SMOOTH_SIGMA > 0:
        m = gaussian_filter(m, SMOOTH_SIGMA)

    top = np.percentile(np.abs(m), 99.5)
    m = m / (top + 1e-12)
    m = np.clip(m, 0 if positive_only else -1, 1)

    m_image = Image.fromarray(m.astype(np.float32), mode="F")
    m_image = m_image.resize(original_image.size, Image.BILINEAR)

    return np.array(m_image)


# All 5 Integrated Gradients display pannrom
plt.figure(figsize=(20, 10))

for i in range(NUM_CLASSES):
    plt.subplot(2, 3, i + 1)

    plt.imshow(original_image)
    plt.imshow(
        to_heatmap(attributions[i]),
        cmap="jet",
        alpha=0.45,
        vmin=0,
        vmax=1
    )

    plt.title(
        f"{CLASS_NAMES[i]}\n"
        f"Probability: {probabilities[i].item():.2%}"
    )
    plt.axis("off")

# Input image display pannrom
plt.subplot(2, 3, 6)

plt.imshow(original_image)
plt.title(
    f"Input Image\n"
    f"True: {CLASS_NAMES[TRUE_LABEL]}\n"
    f"Predicted: {CLASS_NAMES[predicted_class]}"
)
plt.axis("off")

plt.suptitle(
    f"VisionMamba - Integrated Gradients for All 5 Classes "
    f"(baseline={BASELINE}, steps={N_STEPS})",
    fontsize=18
)

plt.tight_layout()
plt.show()

# Predicted class detail: signed attribution (red = pushes towards the class, blue = pushes against it)
signed = to_heatmap(attributions[predicted_class], positive_only=False)
positive = to_heatmap(attributions[predicted_class], positive_only=True)

plt.figure(figsize=(18, 6))

plt.subplot(1, 3, 1)
plt.imshow(original_image)
plt.title(
    f"Input\n"
    f"True: {CLASS_NAMES[TRUE_LABEL]} | Predicted: {CLASS_NAMES[predicted_class]}"
)
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(original_image)
plt.imshow(signed, cmap="bwr", alpha=0.6, vmin=-1, vmax=1)
plt.title("Signed Integrated Gradients\n(red = supports, blue = opposes)")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(original_image)
plt.imshow(positive, cmap="jet", alpha=0.45, vmin=0, vmax=1)
plt.title("Positive Integrated Gradients\n(most important regions)")
plt.axis("off")

plt.suptitle(
    f"VisionMamba - Integrated Gradients for the predicted class: {CLASS_NAMES[predicted_class]}",
    fontsize=16
)

plt.tight_layout()
plt.show()

# Integrated Gradients results print pannrom
print("Integrated Gradients Results - All 5 Classes")
print("True Class     :", CLASS_NAMES[TRUE_LABEL])
print("Predicted Class:", CLASS_NAMES[predicted_class])
print("Confidence     :", f"{predicted_confidence:.4f}")
print("-" * 70)

for i in range(NUM_CLASSES):
    print(f"{CLASS_NAMES[i]:12s} : {probabilities[i].item():.4f}")